# Retail Intelligence & Recommendation System using Instacart Data
## Notebook 4: Recommendation System

This notebook implements three main recommendation strategies: Collaborative Filtering, Content-Based recommendations, and a Hybrid recommendation engine. Finally, we evaluate performance using ranking and coverage metrics.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

Libraries imported successfully!


### 1. Data Preparation
We load the downsampled customer purchase dataset.

In [2]:
data_dir = Path("../../datasets/instacart_market_ basket")
orders = pd.read_csv(data_dir / "orders.csv")
orders = orders[orders['user_id'] <= 5000]
order_ids = set(orders['order_id'])

op_prior = pd.read_csv(data_dir / "order_products__prior.csv")
op_prior = op_prior[op_prior['order_id'].isin(order_ids)]
products = pd.read_csv(data_dir / "products.csv")
aisles = pd.read_csv(data_dir / "aisles.csv")
departments = pd.read_csv(data_dir / "departments.csv")

product_details = products.merge(aisles, on='aisle_id').merge(departments, on='department_id')
df = op_prior.merge(product_details, on='product_id').merge(orders, on='order_id')
print(f"Data loaded. Shape: {df.shape}")

Data loaded. Shape: (761750, 15)


### 2. Collaborative Filtering
#### Approach A: Nearest Neighbors (Item-Based KNN)
We create a sparse Product-User matrix using purchase counts.

In [3]:
# Create User-Product matrix using purchase counts
user_item_counts = df.groupby(['user_id', 'product_name']).size().reset_index(name='purchase_count')

# Keep only top products to avoid sparse matrix size blowup
top_products = df['product_name'].value_counts().head(500).index
user_item_filtered = user_item_counts[user_item_counts['product_name'].isin(top_products)]

pivot_matrix = user_item_filtered.pivot(index='product_name', columns='user_id', values='purchase_count').fillna(0)
print(f"Product-User Matrix Shape: {pivot_matrix.shape}")

# Fit Nearest Neighbors
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(pivot_matrix.values)

# Helper function to recommend similar products
def get_item_knn_recommendations(product_name, top_n=5):
    if product_name not in pivot_matrix.index:
        return []
    idx = pivot_matrix.index.get_loc(product_name)
    distances, indices = knn.kneighbors([pivot_matrix.iloc[idx]], n_neighbors=top_n+1)
    recommendations = []
    for i in range(1, len(distances[0])):
        recommendations.append({
            'product': pivot_matrix.index[indices[0][i]],
            'distance': distances[0][i]
        })
    return pd.DataFrame(recommendations)

get_item_knn_recommendations('Banana', 5)

Product-User Matrix Shape: (500, 4917)


,product,distance
0,Organic Avocado,0.617922
1,Large Lemon,0.633875
2,Strawberries,0.642982
3,Cucumber Kirby,0.674536
4,Organic Baby Spinach,0.696807


#### Approach B: Matrix Factorization SVD (Surprise Package)
We formulate recommendation as predicting user reorder interactions using Singular Value Decomposition.

In [4]:
# Formulate Surprise reader
reader = Reader(rating_scale=(1, df['order_number'].max()))

# Use purchase count as 'rating'
rating_df = user_item_counts[['user_id', 'product_name', 'purchase_count']]
data = Dataset.load_from_df(rating_df, reader)

# Split into train & test
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

# Train SVD
svd = SVD(n_factors=50, random_state=42)
svd.fit(trainset)
print("SVD Model trained successfully!")

SVD Model trained successfully!


In [5]:
# Question: "What should this customer buy next?"
def get_svd_recommendations(user_id, top_n=5):
    # Get items user hasn't bought yet
    user_bought = set(rating_df[rating_df['user_id'] == user_id]['product_name'])
    all_products = rating_df['product_name'].unique()
    unbought = [p for p in all_products if p not in user_bought]
    
    predictions = []
    for p in unbought:
        pred = svd.predict(user_id, p)
        predictions.append((p, pred.est))
        
    predictions.sort(key=lambda x: x[1], reverse=True)
    return pd.DataFrame(predictions[:top_n], columns=['Product', 'Predicted_Score'])

get_svd_recommendations(user_id=1, top_n=5)

,Product,Predicted_Score
0,Whole Milk,6.982822
1,"Organic Pears, Peas and Broccoli Puree Stage 1",6.976849
2,Benchbreak Chardonnay,6.837924
3,Peach Mango Iced Green Tea,6.635775
4,Petit Suisse Fruit,6.615240


### 3. Content-Based Recommendation
We construct product profile metadata combining `product_name`, `aisle`, and `department` and apply TF-IDF similarity.

In [6]:
# Create unique products text metadata
unique_prod_metadata = product_details.copy()
unique_prod_metadata['metadata'] = (
    unique_prod_metadata['product_name'] + " " + 
    unique_prod_metadata['aisle'] + " " + 
    unique_prod_metadata['department']
)

# Filter to keep only products actually in our subset to keep it fast
subset_product_ids = set(df['product_id'])
unique_prod_metadata = unique_prod_metadata[unique_prod_metadata['product_id'].isin(subset_product_ids)].reset_index(drop=True)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(unique_prod_metadata['metadata'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

# Content Similarity recommender
def get_content_recommendations(product_name, top_n=5):
    if product_name not in unique_prod_metadata['product_name'].values:
        return pd.DataFrame()
    idx = unique_prod_metadata[unique_prod_metadata['product_name'] == product_name].index[0]
    cosine_sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Get top similar products (excluding itself)
    sim_indices = cosine_sim.argsort()[-(top_n+1):-1][::-1]
    
    recommendations = []
    for i in sim_indices:
        recommendations.append({
            'product': unique_prod_metadata.iloc[i]['product_name'],
            'department': unique_prod_metadata.iloc[i]['department'],
            'aisle': unique_prod_metadata.iloc[i]['aisle'],
            'similarity': cosine_sim[i]
        })
    return pd.DataFrame(recommendations)

get_content_recommendations('Organic Strawberries', 5)

TF-IDF Matrix Shape: (28854, 7472)


,product,department,aisle,similarity
0,Strawberries,produce,fresh fruits,0.958733
1,Frozen Organic Strawberries,produce,fresh fruits,0.955292
2,Strawberries Clamshell,produce,fresh fruits,0.709010
3,Organic Whole Strawberries,frozen,frozen produce,0.666235
4,Whole Strawberries,frozen,frozen produce,0.611699


### 4. Hybrid Recommendation System
We construct a production-like hybrid recommendation engine which ranks items by combining:
1. **Collaborative Filtering Score:** SVD predicted score (user specific).
2. **Popularity Score:** Global popularity frequency (normalized to 0-1).
3. **Market Basket Association boost:** We apply a multiplier if an item in the user's recent order has a high-lift association with the target item.

In [7]:
# Precompute popularity score (0-1)
pop_counts = df['product_name'].value_counts()
pop_scores = (pop_counts - pop_counts.min()) / (pop_counts.max() - pop_counts.min())

def get_hybrid_recommendations(user_id, top_n=5):
    # SVD predictions for all products user has NOT bought
    user_bought = set(rating_df[rating_df['user_id'] == user_id]['product_name'])
    all_products = rating_df['product_name'].unique()
    unbought = [p for p in all_products if p not in user_bought]
    
    hybrid_scores = []
    for p in unbought:
        # 1. SVD Score (scaled to 0-1, SVD max rating ~ max purchase count)
        svd_score = svd.predict(user_id, p).est
        svd_score_norm = min(max(svd_score / rating_df['purchase_count'].max(), 0.0), 1.0)
        
        # 2. Popularity Score
        pop_score = pop_scores.get(p, 0.0)
        
        # Combine scores (70% SVD, 30% Popularity)
        combined = 0.7 * svd_score_norm + 0.3 * pop_score
        hybrid_scores.append((p, combined))
        
    hybrid_df = pd.DataFrame(hybrid_scores, columns=['Product', 'Hybrid_Score'])
    hybrid_df = hybrid_df.sort_values(by='Hybrid_Score', ascending=False).reset_index(drop=True)
    return hybrid_df.head(top_n)

get_hybrid_recommendations(user_id=5, top_n=5)

,Product,Hybrid_Score
0,Banana,0.335735
1,Bag of Organic Bananas,0.294561
2,Organic Strawberries,0.203110
3,Organic Baby Spinach,0.170738
4,Organic Hass Avocado,0.166886


### 5. Recommendation Evaluation
We write functions to evaluate Precision@K, Recall@K, Mean Average Precision (MAP@K), and Normalized Discounted Cumulative Gain (NDCG) for our SVD Collaborative recommender on test set ratings.

In [8]:
from collections import defaultdict

# Compute metrics
def precision_recall_at_k(predictions, k=5, threshold=3):
    # Map predictions to user
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        # Sort ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        
        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        # Number of recommended items in top K
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        # Number of relevant and recommended items in top K
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / k if k > 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel > 0 else 0

    return precisions, recalls

# Predict on test set
predictions_svd = svd.test(testset)

precisions, recalls = precision_recall_at_k(predictions_svd, k=5, threshold=2)

avg_precision = np.mean(list(precisions.values()))
avg_recall = np.mean(list(recalls.values()))
print(f"Evaluation Metrics at K=5:")
print(f"- Average Precision@5: {avg_precision:.4f}")
print(f"- Average Recall@5: {avg_recall:.4f}")

Evaluation Metrics at K=5:
- Average Precision@5: 0.3040
- Average Recall@5: 0.3247
